stacking Recurrent layers
xếp chồng các lớp RNN

why?
giúp mạng lưới học các đặc trưng rõ hơn
tầng 1: có dữ liệu thô => học được các quy luật cơ bản, ngắn hạn
tầng 2: chỉ có đầu vào là trạng thái ẩn từ tầng 1 khi đó sẽ nắm bắt được mọi thứ từ dưới lên hiểu được khải niệm trừu tượng, dài hạn hơn.

nguyên tắc xây dựng nhiều lớp 
lớp 1 bắt buộc nhả ra toàn bộ cuộn băng ghi hình (chính là cái output trong pytorch) để là đầu vào cho lớp thứ 2

lớp 2 chỉ cần nhả ra cái chốt sổ cuối cùng (hn) hoặc nếu xây nhiều lớp thì nhả ra cái khác tùy

lưu ý:
không cần xâu nhà quá cao 2, 3 tầng là kịch kim ko bị vanishing hoặc overfitting vs tốn tài nguyên

cách 1 xây thủ công 

linh hoạt trong việc hiểu kết hợp đc cả LSTM + GRU

nhưng chậm hơn vì ko hỗ trợ chạy trên card đồ họa đc

In [ ]:
import torch
import torch.nn as nn


class StackedRNN(nn.Module):
    def __init__(self, input_size, hidden_size1, hidden_size2, output_size ):
        super().__init__()

        # khởi tạo 2 cái lớp này

        self.lstm1 = nn.LSTM(input_size, hidden_size1, batch_first= True)

        self.gru2 = nn.GRU(hidden_size1, hidden_size2, batch_first= True)

        self.fc = nn.Linear(hidden_size2, output_size)

    def forward(self, x):
        
        output_seq1, final_state1 = self.lstm1(x)

        output_seq2, final_state2 = self.gru2(output_seq1)

        last_time_step_output = output_seq2[:, -1, :]

        out = self.fc(last_time_step_output)

        return out


cách 2 dùng lệnh num_layers

rất nhanh nhưng khongo trộn 2 cái khác nhau vô đc

In [ ]:
import torch 
import torch.nn as nn

class StackedRNNInternal(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super().__init__()

        self.lstm = nn.LSTM(input_size, hidden_size,
                            num_layers= num_layers, batch_first= True)
        
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        output, (h_n, c_n) = self.lstm(x)

        last_time_step_output  = output[:, -1, :]

        out = self.fc(last_time_step_output)

        return out
